In [2]:
import pandas as pd

In [3]:
%pip install seaborn

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install fg-data-profiling "setuptools<81"

Note: you may need to restart the kernel to use updated packages.


In [6]:
import data_profiling
from data_profiling import ProfileReport

d:\Work\ExcelAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
horse_health_train = pd.read_csv("datasets/horse_health/train.csv")
horse_health_test = pd.read_csv("datasets/horse_health/test.csv")
# profile = ProfileReport(horse_health_train, title="Horse Health Train Data Profiling Report")
# profile.to_file("horse_health_train_profile.html")

In [8]:
horse_health_train_raw = pd.read_csv("datasets/horse_health/train.csv")
horse_health_test_raw = pd.read_csv("datasets/horse_health/test.csv")

horse_health_outcome = horse_health_train_raw["outcome"].astype('category')
horse_health_outcome_labels = horse_health_outcome.cat.categories
horse_health_outcome = horse_health_outcome.cat.codes
horse_health_train = horse_health_train_raw.drop(columns=["id", "hospital_number", "lesion_2", "lesion_3", "outcome"], errors="ignore")
horse_health_test = horse_health_test_raw.drop(columns=["id", "hospital_number", "lesion_2", "lesion_3"], errors="ignore")

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

In [10]:
numeric_features = horse_health_train.select_dtypes(exclude='object').columns
categorical_features = horse_health_train.select_dtypes(include='object').columns

preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_features)
])

horse_model_lgbm = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LGBMClassifier(random_state=42, n_estimators=500, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, n_jobs=-1))
])
horse_model_xgb = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(random_state=42, n_estimators=500, learning_rate=0.03, max_depth=4, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, eval_metric='mlogloss'))
])

horse_model_random_forest = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=500, max_depth=10, min_samples_split=5, min_samples_leaf=2, n_jobs=-1))
])


In [11]:
X_train, X_valid, y_train, y_valid = train_test_split(horse_health_train, horse_health_outcome, test_size=0.2, random_state=42, stratify=horse_health_outcome)

In [12]:
horse_model_lgbm.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000855 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 431
[LightGBM] [Info] Number of data points in the train set: 988, number of used features: 61
[LightGBM] [Info] Start training from score -1.102669
[LightGBM] [Info] Start training from score -1.592378
[LightGBM] [Info] Start training from score -0.766632


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [13]:
score = horse_model_lgbm.score(X_valid, y_valid)
print(f"LightGBM Validation Accuracy: {score:.4f}")

LightGBM Validation Accuracy: 0.7126


d:\Work\ExcelAI\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [14]:
horse_model_xgb.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [15]:
print(f"XGBoost Validation Accuracy: {horse_model_xgb.score(X_valid, y_valid):.4f}")

XGBoost Validation Accuracy: 0.7126


In [16]:
horse_model_random_forest.fit(X_train, y_train)
print(f"Random Forest Validation Accuracy: {horse_model_random_forest.score(X_valid, y_valid):.4f}")


Random Forest Validation Accuracy: 0.6802


In [17]:
target_path = "datasets/horse_health/sample_submission.csv"

In [18]:
from sklearn.ensemble import VotingClassifier

In [19]:
model = VotingClassifier(estimators=[("lgbm", horse_model_lgbm), ("xgb", horse_model_xgb), ("rf", horse_model_random_forest)], voting="soft")
model.fit(X_train, y_train)
predicted_outcomes = model.predict(horse_health_test)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000277 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 431
[LightGBM] [Info] Number of data points in the train set: 988, number of used features: 61
[LightGBM] [Info] Start training from score -1.102669
[LightGBM] [Info] Start training from score -1.592378
[LightGBM] [Info] Start training from score -0.766632


d:\Work\ExcelAI\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [20]:
transformed_outcomes = [horse_health_outcome_labels[pred] for pred in predicted_outcomes]
submission_df = pd.DataFrame({
    "id": horse_health_test_raw["id"],
    "outcome": transformed_outcomes
})
submission_df.to_csv(target_path, index=False)

In [21]:
cirrhosis_train_raw = pd.read_csv("datasets/cirrhosis/train.csv")
cirrhosis_test_raw = pd.read_csv("datasets/cirrhosis/test.csv")

cirrhosis_train = cirrhosis_train_raw.drop(columns=["Status", "id"], errors="ignore")
cirrhosis_test = cirrhosis_test_raw.drop(columns=["id"], errors="ignore")
cirrhosis_outcome = cirrhosis_train_raw["Status"].astype('category')
cirrhosis_outcome_labels = cirrhosis_outcome.cat.categories
cirrhosis_outcome = cirrhosis_outcome.cat.codes

In [22]:
profile = ProfileReport(cirrhosis_train_raw, title="Cirrhosis Train Data Profiling Report")
profile.to_file("cirrhosis_train_profile.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 29.97it/s]


In [23]:
cirrhosis_categorical_features = cirrhosis_train.select_dtypes(include='object').columns
cirrhosis_numerical_features = cirrhosis_train.select_dtypes(exclude='object').columns

X_train_cirrhosis, X_valid_cirrhosis, y_train_cirrhosis, y_valid_cirrhosis = train_test_split(cirrhosis_train, cirrhosis_outcome, test_size=0.2, random_state=42, stratify=cirrhosis_outcome)

preprocessor_cirrhosis = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), cirrhosis_numerical_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cirrhosis_categorical_features)
])

cirrhosis_lgbm_model = Pipeline([
    ('preprocessor', preprocessor_cirrhosis),
    ('classifier', LGBMClassifier(random_state=42, n_estimators=500, learning_rate=0.03, num_leaves=31, n_jobs=-1))
])

cirrhosis_xgboost_model = Pipeline([
    ('preprocessor', preprocessor_cirrhosis),
    ('classifier', XGBClassifier(random_state=42, n_estimators=500, learning_rate=0.03, max_depth=4, n_jobs=-1, eval_metric='mlogloss'))
])

cirrhosis_rf_model = Pipeline([
    ('preprocessor', preprocessor_cirrhosis),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=500, max_depth=10, min_samples_split=5, min_samples_leaf=2, n_jobs=-1))
])

In [24]:
#LGBM stats
lgbm_acc = cirrhosis_lgbm_model.fit(X_train_cirrhosis, y_train_cirrhosis).score(X_valid_cirrhosis, y_valid_cirrhosis)
print(f"Cirrhosis LGBM Validation Accuracy: {lgbm_acc:.4f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000920 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1937
[LightGBM] [Info] Number of data points in the train set: 6324, number of used features: 25
[LightGBM] [Info] Start training from score -0.465082
[LightGBM] [Info] Start training from score -3.358480
[LightGBM] [Info] Start training from score -1.087291
Cirrhosis LGBM Validation Accuracy: 0.8305


d:\Work\ExcelAI\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [25]:
#XGBoost stats
xgb_acc = cirrhosis_xgboost_model.fit(X_train_cirrhosis, y_train_cirrhosis).score(X_valid_cirrhosis, y_valid_cirrhosis)
print(f"Cirrhosis XGBoost Validation Accuracy: {xgb_acc:.4f}")

Cirrhosis XGBoost Validation Accuracy: 0.8292


In [26]:
#RF stats
rf_acc = cirrhosis_rf_model.fit(X_train_cirrhosis, y_train_cirrhosis).score(X_valid_cirrhosis, y_valid_cirrhosis)
print(f"Cirrhosis Random Forest Validation Accuracy: {rf_acc:.4f}")

Cirrhosis Random Forest Validation Accuracy: 0.8248


In [27]:
model_cirrhosis = VotingClassifier(estimators=[("lgbm", cirrhosis_lgbm_model), ("xgb", cirrhosis_xgboost_model), ("rf", cirrhosis_rf_model)], voting="soft")
model_cirrhosis.fit(cirrhosis_train, cirrhosis_outcome)
cirrhosis_probas = model_cirrhosis.predict_proba(cirrhosis_test)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001441 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1942
[LightGBM] [Info] Number of data points in the train set: 7905, number of used features: 25
[LightGBM] [Info] Start training from score -0.465082
[LightGBM] [Info] Start training from score -3.358480
[LightGBM] [Info] Start training from score -1.087291


d:\Work\ExcelAI\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [28]:
submission_cirrhosis_df = pd.DataFrame({
    "id": cirrhosis_test_raw["id"],
    "Status_C": cirrhosis_probas[:, 0].round(6),
    "Status_CL": cirrhosis_probas[:, 1].round(6),
    "Status_D": cirrhosis_probas[:, 2].round(6)
})

submission_cirrhosis_df.to_csv("datasets/cirrhosis/sample_submission.csv", index=False)

In [29]:
thapar_path = "datasets/thapar"
import os

In [35]:
thapar_train_raw = pd.read_csv(os.path.join(thapar_path, "train.csv"))
thapar_test_raw = pd.read_csv(os.path.join(thapar_path, "test.csv"))

thapar_train = thapar_train_raw.drop(columns=["id", "Status"], errors="ignore")
thapar_outcome = thapar_train_raw["Status"].astype('category')
thapar_outcome_labels = thapar_outcome.cat.categories
thapar_outcome = thapar_outcome.cat.codes
thapar_test = thapar_test_raw.drop(columns=["id"], errors="ignore")


In [34]:

profile = ProfileReport(thapar_train_raw, title="Thapar Train Data Profiling Report")
profile.to_file("thapar_train_profile.html")


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 32.59it/s]


In [36]:
thapar_categorical_features = thapar_train.select_dtypes(include='object').columns
thapar_numerical_features = thapar_train.select_dtypes(exclude='object').columns

X_train_thapar, X_valid_thapar, y_train_thapar, y_valid_thapar = train_test_split(thapar_train, thapar_outcome, test_size=0.2, random_state=42, stratify=thapar_outcome)

preprocessor_thapar = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), thapar_numerical_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), thapar_categorical_features)
])

thapar_lgbm_model = Pipeline([
    ('preprocessor', preprocessor_thapar),
    ('classifier', LGBMClassifier(random_state=42, n_estimators=500, learning_rate=0.03, num_leaves=31, n_jobs=-1))
])

thapar_xgboost_model = Pipeline([
    ('preprocessor', preprocessor_thapar),
    ('classifier', XGBClassifier(random_state=42, n_estimators=500, learning_rate=0.03, max_depth=4, n_jobs=-1, eval_metric='mlogloss'))
])

thapar_rf_model = Pipeline([
    ('preprocessor', preprocessor_thapar),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=500, max_depth=10, min_samples_split=5, min_samples_leaf=2, n_jobs=-1))
])

In [37]:
#LGBM stats
lgbm_acc_thapar = thapar_lgbm_model.fit(X_train_thapar, y_train_thapar).score(X_valid_thapar, y_valid_thapar)
print(f"Thapar LGBM Validation Accuracy: {lgbm_acc_thapar:.4f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001545 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1937
[LightGBM] [Info] Number of data points in the train set: 6324, number of used features: 25
[LightGBM] [Info] Start training from score -0.465082
[LightGBM] [Info] Start training from score -3.358480
[LightGBM] [Info] Start training from score -1.087291
Thapar LGBM Validation Accuracy: 0.8305


d:\Work\ExcelAI\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [38]:
#xgboost stats
xgb_acc_thapar = thapar_xgboost_model.fit(X_train_thapar, y_train_thapar).score(X_valid_thapar, y_valid_thapar)
print(f"Thapar XGBoost Validation Accuracy: {xgb_acc_thapar:.4f}")

Thapar XGBoost Validation Accuracy: 0.8292


In [39]:
#RF stats
rf_acc_thapar = thapar_rf_model.fit(X_train_thapar, y_train_thapar).score(X_valid_thapar, y_valid_thapar)
print(f"Thapar Random Forest Validation Accuracy: {rf_acc_thapar:.4f}")

Thapar Random Forest Validation Accuracy: 0.8248


In [44]:
thapar_model = VotingClassifier(estimators=[("lgbm", thapar_lgbm_model), ("xgb", thapar_xgboost_model), ("rf", thapar_rf_model)], voting="soft")
thapar_model.fit(thapar_train, thapar_outcome)

#model acc
print(f"Thapar Ensemble Validation Accuracy: {thapar_model.score(X_valid_thapar, y_valid_thapar):.4f}")

predicted_outcomes_thapar = thapar_model.predict_proba(thapar_test)

submission_thapar_df = pd.DataFrame({
    "id": thapar_test_raw["id"],
    "Status_C": predicted_outcomes_thapar[:, 0],
    "Status_CL": predicted_outcomes_thapar[:, 1],
    "Status_D": predicted_outcomes_thapar[:, 2]
})
submission_thapar_df.to_csv("datasets/thapar/sample_submission.csv", index=False)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001265 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1942
[LightGBM] [Info] Number of data points in the train set: 7905, number of used features: 25
[LightGBM] [Info] Start training from score -0.465082
[LightGBM] [Info] Start training from score -3.358480
[LightGBM] [Info] Start training from score -1.087291


d:\Work\ExcelAI\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Thapar Ensemble Validation Accuracy: 0.9222


d:\Work\ExcelAI\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [48]:
monster_path = "datasets/monsters"
monster_raw_train = pd.read_csv(os.path.join(monster_path, "train.csv"))
monster_raw_test = pd.read_csv(os.path.join(monster_path, "test.csv"))

monster_train = monster_raw_train.drop(columns=["id", "type"], errors="ignore")
monster_outcome = monster_raw_train["type"].astype('category')
monster_outcome_labels = monster_outcome.cat.categories
monster_outcome = monster_outcome.cat.codes

monster_test = monster_raw_test.drop(columns=["id"], errors="ignore")

In [49]:
profile_monster = ProfileReport(monster_raw_train, title="Monster Train Data Profiling Report")
profile_monster.to_file("monster_train_profile.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 94.56it/s]


In [51]:
preprocessor_monster = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), monster_train.select_dtypes(exclude='object').columns)
])

X_train_monster, X_valid_monster, y_train_monster, y_valid_monster = train_test_split(monster_train, monster_outcome, test_size=0.2, random_state=42, stratify=monster_outcome)

monster_lgbm_model = Pipeline([
    ('preprocessor', preprocessor_monster),
    ('classifier', LGBMClassifier(random_state=42, n_estimators=500, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, n_jobs=-1))
])

monster_xgboost_model = Pipeline([
    ('preprocessor', preprocessor_monster),
    ('classifier', XGBClassifier(random_state=42, n_estimators=500, learning_rate=0.03, max_depth=4, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, eval_metric='mlogloss'))
])

monster_rf_model = Pipeline([
    ('preprocessor', preprocessor_monster),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=500, max_depth=10, min_samples_split=5, min_samples_leaf=2, n_jobs=-1))
])


In [52]:
#LGBM stats
lgbm_acc_monster = monster_lgbm_model.fit(monster_train, monster_outcome).score(monster_train, monster_outcome)
print(f"Monster LGBM Training Accuracy: {lgbm_acc_monster:.4f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000152 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 496
[LightGBM] [Info] Number of data points in the train set: 371, number of used features: 4
[LightGBM] [Info] Start training from score -1.154028
[LightGBM] [Info] Start training from score -1.056390
[LightGBM] [Info] Start training from score -1.087888
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furt

d:\Work\ExcelAI\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [53]:
#XGBoost stats
xgb_acc_monster = monster_xgboost_model.fit(monster_train, monster_outcome).score(monster_train, monster_outcome)
print(f"Monster XGBoost Training Accuracy: {xgb_acc_monster:.4f}")

Monster XGBoost Training Accuracy: 1.0000


In [54]:
#RF stats
rf_acc_monster = monster_rf_model.fit(monster_train, monster_outcome).score(monster_train, monster_outcome)
print(f"Monster Random Forest Training Accuracy: {rf_acc_monster:.4f}")

Monster Random Forest Training Accuracy: 0.9623


In [56]:
monster_model = VotingClassifier(estimators=[("lgbm", monster_lgbm_model), ("xgb", monster_xgboost_model), ("rf", monster_rf_model)], voting="soft")
monster_model.fit(monster_train, monster_outcome)

predicted_outcomes_monster = monster_model.predict(monster_test)
predicted_outcomes_monster_labels = [monster_outcome_labels[pred] for pred in predicted_outcomes_monster]
submission_monster_df = pd.DataFrame({
    "id": monster_raw_test["id"],
    "type": predicted_outcomes_monster_labels
})
submission_monster_df.to_csv("datasets/monsters/sample_submission.csv", index=False)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000126 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 496
[LightGBM] [Info] Number of data points in the train set: 371, number of used features: 4
[LightGBM] [Info] Start training from score -1.154028
[LightGBM] [Info] Start training from score -1.056390
[LightGBM] [Info] Start training from score -1.087888
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furt

d:\Work\ExcelAI\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [62]:
obesity_path = "datasets/obsesity"
obesity_train_raw = pd.read_csv(os.path.join(obesity_path, "train.csv"))
obesity_test_raw = pd.read_csv(os.path.join(obesity_path, "test.csv"))

obesity_train = obesity_train_raw.drop(columns=["id", "NObeyesdad"], errors="ignore")
obesity_test = obesity_test_raw.drop(columns=["id"], errors="ignore")
obesity_outcome = obesity_train_raw["NObeyesdad"].astype('category')
obesity_outcome_labels = obesity_outcome.cat.categories
obesity_outcome = obesity_outcome.cat.codes

In [65]:
obesity_numerical_features = obesity_train.select_dtypes(exclude='object').columns
obesity_cathegorical_features = obesity_train.select_dtypes(include='object').columns

X_train_obesity, X_valid_obesity, y_train_obesity, y_valid_obesity = train_test_split(obesity_train, obesity_outcome, test_size=0.2, random_state=42, stratify=obesity_outcome)

preprocessor_obesity = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), obesity_numerical_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), obesity_cathegorical_features)
])

obesity_lgbm_model = Pipeline([
    ('preprocessor', preprocessor_obesity),
    ('classifier', LGBMClassifier(random_state=42, n_estimators=500, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, n_jobs=-1))
])

obesity_xgboost_model = Pipeline([
    ('preprocessor', preprocessor_obesity),
    ('classifier', XGBClassifier(random_state=42, n_estimators=500, learning_rate=0.03, max_depth=4, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, eval_metric='mlogloss'))
])

obesity_rf_model = Pipeline([
    ('preprocessor', preprocessor_obesity),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=500, max_depth=10, min_samples_split=5, min_samples_leaf=2, n_jobs=-1))
])

In [66]:
#LGBM stats
lgbm_acc_obesity = obesity_lgbm_model.fit(X_train_obesity, y_train_obesity).score(X_valid_obesity, y_valid_obesity)
print(f"Obesity LGBM Validation Accuracy: {lgbm_acc_obesity:.4f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001015 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2070
[LightGBM] [Info] Number of data points in the train set: 16606, number of used features: 30
[LightGBM] [Info] Start training from score -2.107657
[LightGBM] [Info] Start training from score -1.907572
[LightGBM] [Info] Start training from score -1.964755
[LightGBM] [Info] Start training from score -1.855022
[LightGBM] [Info] Start training from score -1.635117
[LightGBM] [Info] Start training from score -2.146046
[LightGBM] [Info] Start training from score -2.107657
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

d:\Work\ExcelAI\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [67]:
#XGBoost stats
xgb_acc_monster = monster_xgboost_model.fit(monster_train, monster_outcome).score(monster_train, monster_outcome)
print(f"Monster XGBoost Training Accuracy: {xgb_acc_monster:.4f}")

Monster XGBoost Training Accuracy: 1.0000


In [68]:
#RF stats
rf_acc_monster = monster_rf_model.fit(monster_train, monster_outcome).score(monster_train, monster_outcome)
print(f"Monster Random Forest Training Accuracy: {rf_acc_monster:.4f}")

Monster Random Forest Training Accuracy: 0.9623


In [70]:
obesity_model = VotingClassifier(estimators=[("lgbm", obesity_lgbm_model), ("xgb", obesity_xgboost_model), ("rf", obesity_rf_model)], voting="soft")
obesity_model.fit(obesity_train, obesity_outcome)

predicted_outcomes_obesity = obesity_model.predict(obesity_test)
predicted_outcomes_obesity_labels = [obesity_outcome_labels[pred] for pred in predicted_outcomes_obesity]

submission_obesity_df = pd.DataFrame({
    "id": obesity_test_raw["id"],
    "NObeyesdad": predicted_outcomes_obesity_labels
})

submission_obesity_df.to_csv("datasets/obsesity/sample_submission.csv", index=False)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002201 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2068
[LightGBM] [Info] Number of data points in the train set: 20758, number of used features: 30
[LightGBM] [Info] Start training from score -2.107483
[LightGBM] [Info] Start training from score -1.907353
[LightGBM] [Info] Start training from score -1.964779
[LightGBM] [Info] Start training from score -1.854892
[LightGBM] [Info] Start training from score -1.635203
[LightGBM] [Info] Start training from score -2.146276
[LightGBM] [Info] Start training from score -2.107879


d:\Work\ExcelAI\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
